# Activation Oracle via the vllm-lens Inspect Provider

This notebook demonstrates the **activation oracle** using the `vllm-lens`
[Inspect AI](https://inspect.aisi.org.uk/) model provider, structured as an
Inspect **solver** so it can be composed into evaluation tasks.

The oracle (from [LatentQA / Activation Oracles](https://arxiv.org/abs/2412.08586))
lets you "ask questions" about a model's internal activations.  A LoRA adapter
interprets steering vectors injected at a special token, enabling the model to
describe what it is "thinking about" at a given layer and token.

**Flow (inside the solver):**
1. Capture residual-stream activations from a target prompt.
2. Extract the activation at a token of interest (e.g. "pupil").
3. Inject it into an oracle prompt via steering and generate with the oracle LoRA.

## Setup

In [1]:
from inspect_ai import Task, eval
from inspect_ai.dataset import MemoryDataset, Sample
from inspect_ai.model import (
    ChatMessageAssistant,
    ChatMessageUser,
    GenerateConfig,
    ChatMessage,
    get_model,
)
from inspect_ai.solver import Generate, TaskState, solver
from transformers import AutoTokenizer

from vllm_lens import SteeringVector

In [2]:
MODEL = "Qwen/Qwen3-8B"
LORA_PATH = "adamkarvonen/checkpoints_latentqa_cls_past_lens_addition_Qwen3-8B"
TARGET_CONTENT = (
    "The philosopher who drank hemlock taught a student who founded "
    "an academy. That student's most famous pupil was"
)
ORACLE_QUESTION = "Can you name all people the model is currently thinking about?"

## Start the Model and Capture Activations

Start the vLLM server once with LoRA support enabled, then run a quick
activation capture to verify the residual stream extraction works.

In [3]:
model = get_model(
    f"vllm-lens/{MODEL}",
    enable_lora=True,
    max_lora_rank=64,
    max_model_len=4096,
    gpu_memory_utilization=0.85,
    lora_modules=f"oracle={LORA_PATH}",
)
capture_config = GenerateConfig(
    temperature=0.0,
    max_tokens=1,
    extra_body={
        "extra_args": {"output_residual_stream": [9, 18, 27]},
        "chat_template_kwargs": {"enable_thinking": False},
    },
)
output = await model.generate(
    [ChatMessageAssistant(content=TARGET_CONTENT)], config=capture_config
)
residual_stream = output.metadata["activations"]["residual_stream"]  # type: ignore
print("residual_stream shape:", residual_stream.shape)

residual_stream shape: torch.Size([3, 24, 4096])


## Define the Activation Oracle Solver

The solver extracts the target text from the sample's assistant message,
captures residual-stream activations, locates the token of interest, and
then queries the oracle LoRA at each captured layer.

In [4]:
@solver
def ask_activation_oracle(
    oracle_question: str,
    token_of_interest: str,
    lora_path: str,
    extraction_activation_layers: list[int] = [9, 18, 27],
    injection_layer: int = 1,
    injection_special_token: str = " ?",
    steering_coefficient: float = 1.0,
):
    tokenizer = AutoTokenizer.from_pretrained(MODEL)

    async def solve(state: TaskState, _solver_generate: Generate) -> TaskState:
        # Extract activations from the forward pass
        print(state.messages)
        model = get_model()
        capture_config = GenerateConfig(
            temperature=0.0,
            max_tokens=1,
            extra_body={
                "extra_args": {"output_residual_stream": extraction_activation_layers},
                "chat_template_kwargs": {"enable_thinking": False},
            },
        )
        output = await model.generate(state.messages, config=capture_config)
        residual_stream = output.metadata["activations"]["residual_stream"]
        print("residual_stream shape:", residual_stream.shape)

        # --- Step 2: Find token of interest and extract activations ---
        target_prompt = tokenizer.apply_chat_template(
            state.messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        target_token_ids = tokenizer.encode(target_prompt, add_special_tokens=False)

        tok_candidates = tokenizer.encode(
            f" {token_of_interest}", add_special_tokens=False
        )
        tok_id = tok_candidates[-1]

        tok_pos = None
        for i, tid in enumerate(target_token_ids):
            if tid == tok_id:
                tok_pos = i
        assert tok_pos is not None, f"Could not find '{token_of_interest}' token"

        token_acts = {}
        for layer_i, layer in enumerate(extraction_activation_layers):
            token_acts[layer] = residual_stream[layer_i, tok_pos, :]

        # --- Step 3: Run oracle at each layer ---
        oracle_results = []
        for layer in extraction_activation_layers:
            prefix = f"Layer: {layer}\n{injection_special_token} \n"
            oracle_content = prefix + oracle_question

            oracle_messages = [{"role": "user", "content": oracle_content}]
            oracle_prompt = tokenizer.apply_chat_template(
                oracle_messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
            oracle_token_ids = tokenizer.encode(oracle_prompt, add_special_tokens=False)

            special_token_id = tokenizer.encode(
                injection_special_token, add_special_tokens=False
            )
            assert len(special_token_id) == 1
            special_pos = oracle_token_ids.index(special_token_id[0])

            act_vector = token_acts[layer].unsqueeze(0).unsqueeze(0)

            messages: list[ChatMessage] = [ChatMessageUser(content=oracle_content)]

            oracle_config = GenerateConfig(
                temperature=0.0,
                max_tokens=50,
                extra_body={
                    "extra_args": {
                        "apply_steering_vectors": [
                            SteeringVector(
                                activations=act_vector,
                                layer_indices=[injection_layer],
                                scale=steering_coefficient,
                                norm_match=True,
                                position_indices=[special_pos],
                            )
                        ],
                    },
                    "lora_request": {
                        "lora_name": "oracle",
                        "lora_int_id": 1,
                        "lora_path": lora_path,
                    },
                    "chat_template_kwargs": {"enable_thinking": False},
                },
            )

            response = await model.generate(messages, config=oracle_config)

            oracle_results.append(
                {
                    "layer": layer,
                    "activation_norm": float(token_acts[layer].norm().item()),
                    "response": response.completion,
                }
            )

        state.metadata["oracle_results"] = oracle_results
        return state

    return solve

## Set Up Task and Run

Create a `Task` with a single `Sample` whose input is the target content
(as an assistant message), wire up the solver, and run with `eval()`.
The model started above is reused for the evaluation.

In [ ]:
sample = Sample(
    input=[ChatMessageAssistant(content=TARGET_CONTENT)],
    target="oracle",
)

task = Task(
    dataset=MemoryDataset([sample]),
    solver=ask_activation_oracle(
        oracle_question=ORACLE_QUESTION,
        token_of_interest="pupil",
        lora_path=LORA_PATH,
    ),  # type: ignore
)

log = eval(task, model=model, display="none")[0]

Output()

## Results

In [ ]:
print(f"Status: {log.status}\n")

for result in log.samples[0].metadata["oracle_results"]:  # type: ignore
    layer = result["layer"]
    norm = result["activation_norm"]
    response = result["response"]
    print(f"=== Layer {layer} (activation norm: {norm:.2f}) ===")
    print(f"Oracle: {response}\n")

Status: success

=== Layer 9 (activation norm: 69.50) ===
Oracle: The assistant is currently thinking about the philosopher NAME_1.

=== Layer 18 (activation norm: 110.50) ===
Oracle: The model is currently thinking about Socrates, Plato, and Aristotle.

=== Layer 27 (activation norm: 368.00) ===
Oracle: The assistant is currently thinking about the philosopher NAME_1.

